In [63]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    classification_report,
)

# Load dataset
dataset = load_breast_cancer()
X = dataset.data
y = dataset.target

# Fixed train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) 

# Experiment controls
sample_size = 20
features_size = 4

# Select features using training data only
selector = SelectKBest(score_func=f_classif, k=features_size)

X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

# Display selected features
selected_features = dataset.feature_names[selector.get_support()]
print("Selected features:")
for feature in selected_features:
    print("-", feature)

# Scale from 0 to pi
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)

# Apply the same sample selection to both models
X_train_subset = X_train_scaled[:sample_size]
y_train_subset = y_train[:sample_size]

# Keep the full test set for evaluation
X_test_subset = X_test_scaled
y_test_subset = y_test

Selected features:
- mean concave points
- worst radius
- worst perimeter
- worst concave points


In [64]:
classical_svm = SVC(kernel="rbf")
classical_svm.fit(X_train_subset, y_train_subset)

y_pred_classical = classical_svm.predict(X_test_subset)

print("Classical SVM accuracy:",
      accuracy_score(y_test_subset, y_pred_classical))

print(classification_report(
    y_test_subset,
    y_pred_classical,
    target_names=dataset.target_names,
    zero_division=0,
))

print("Malignant recall:",
      recall_score(y_test_subset, y_pred_classical, pos_label=0))

Classical SVM accuracy: 0.9473684210526315
              precision    recall  f1-score   support

   malignant       0.97      0.88      0.93        42
      benign       0.93      0.99      0.96        72

    accuracy                           0.95       114
   macro avg       0.95      0.93      0.94       114
weighted avg       0.95      0.95      0.95       114

Malignant recall: 0.8809523809523809
